We downloaded List_doc.xlsx from https://basedoc.diplomatie.gouv.fr/pages/traites/ using period filters

In [1]:
import pandas as pd 

In [11]:
tab_doc['Titre'].unique()

array(["Déclaration d'intention entre le ministre des affaires étrangères et du développement international de la République française et le ministre des affaires étrangères de l'Etat du Qatar relatif à la coopération dans l'organisation de la coupe du monde de Football 2022",
       "Déclaration d'intention entre la ministre de la culture et de la communication de la République française et la ministre de la culture et de la sauvegarde du patrimoine de la République tunisienne sur le renforcement de la coopération dans le domaine culturel",
       "Déclaration d'intention entre le ministre des affaires étrangères et du développement international de la République française et le ministre des affaires étrangères de la République tunisienne relative au développement régional",
       "Déclaration d'intention dans le domaine du tourisme entre le secrétaire d'État chargé du commerce extérieur, de la promotion du tourisme et des Français de l'étranger de la République française et la minis

In [2]:
tab_doc = pd.read_csv("List_doc.csv", sep=";", encoding="latin-1")
tab_doc.head()

,MAE_EFFET_MOD,MAE_COND_DUR,MAE_DATE_VIG_FR,MAE_EFFET_MODPAR,DOC_ORIG_REF,Nature de l'accord,Lieu de signature,Date d'adoption,MAE_BIBLIO,MAE_DATE_DENONC,...,DOC_AUTEUR,MAE_FONCTION,MAE_COMM_CHOI,DOC_ATTACHE,MAE_FONCTION2,DOC_ORIG_TITRE,Thème,MAE_DUREE,MAE_RES_FR,doc_autmoral_sans_accent
0,NaN,NaN,NaN,NaN,NaN,BILATERAL,Doha,2015-04-05,NaN,NaN,...,NaN,NaN,NaN,NaN,ministre des affaires étrangères,NaN,sport,NaN,NaN,"qatar. ministere des affaires etrangeres, fran..."
1,NaN,NaN,NaN,NaN,NaN,BILATERAL,Paris,2015-07-04,NaN,NaN,...,NaN,NaN,NaN,NaN,ministre de la culture et de la sauvegarde du ...,NaN,culture,NaN,NaN,tunisie. ministere de la culture et de la sauv...
2,NaN,NaN,NaN,NaN,NaN,BILATERAL,Paris,2015-07-04,NaN,NaN,...,NaN,NaN,NaN,NaN,ministre des affaires étrangères,NaN,aide au développement,NaN,NaN,"tunisie. ministere des affaires etrangeres, fr..."
3,NaN,NaN,NaN,NaN,NaN,BILATERAL,Paris,2015-01-26,NaN,NaN,...,NaN,NaN,NaN,NaN,"ministre du commerce, de l'insdustrie et du to...",NaN,tourisme,NaN,NaN,colombie. ministere de l'agriculture et du dev...
4,NaN,NaN,NaN,NaN,NaN,BILATERAL,Bogota,2015-06-25,NaN,NaN,...,NaN,NaN,NaN,NaN,"ambassadeur de France en Colombie, ministre de...",NaN,enseignement supérieur,NaN,NaN,"france. ministere de l'education nationale, de..."


In [3]:
# We drop each column with more than 90% of Nan
tab_doc = tab_doc.loc[:, tab_doc.isna().mean() < 0.9]
tab_doc.columns

Index(['MAE_COND_DUR', 'DOC_ORIG_REF', 'Nature de l'accord',
       'Lieu de signature', 'Date d'adoption', 'MAE_FONCSSPV',
       'Type juridique', 'MAE_CONFID2', 'MAE_DATE_VIG', 'MAE_CLASSIF',
       'DOC_AUTMORAL', 'MAE_VIGUEUR', 'MAE_COND_VIG', 'Titre', 'DOC_CONGPAYS',
       'Cote', 'Dépositaire', 'MAE_AUTSSPV', 'MAE_AUT_TEMP2',
       'Date de signature', 'MAE_FONCTION', 'DOC_ATTACHE', 'MAE_FONCTION2',
       'DOC_ORIG_TITRE', 'Thème', 'doc_autmoral_sans_accent'],
      dtype='object')

## Get Data

The links are based on ```Cote``` column

In [4]:
base_url = "https://basedoc.diplomatie.gouv.fr/pages/details-traites/?refine.doc_ref="

# For instance, we can get the url of the first document with the following code:
url_doc_1 = base_url + tab_doc.loc[1, "Cote"]
import requests
from bs4 import BeautifulSoup
response = requests.get(url_doc_1)
soup = BeautifulSoup(response.text, "html.parser")


In [5]:
# Approche simple avec Selenium standard + headers et délais
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import random

# Configuration du driver avec options anti-détection
options = Options()
options.add_argument('--headless')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# Initialiser le driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Masquer que nous sommes un bot
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

try:
    print(f"Navigation vers {url_doc_1}...")
    driver.get(url_doc_1)
    
    # Délai aléatoire humain-like
    wait_time = random.uniform(4, 8)
    print(f"Attente de {wait_time:.1f} secondes (comme un humain)...")
    time.sleep(wait_time)
    
    # Scroll un peu vers le bas (comportement humain)
    driver.execute_script("window.scrollTo(0, 300);")
    time.sleep(random.uniform(0.5, 1.5))
    
    print("Page chargée, recherche de la section Documents...")
    
    # Attendre que la section Documents soit présente
    wait = WebDriverWait(driver, 15)
    documents_section = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "section[aria-labelledby='documents-title']"))
    )
    print("✓ Section Documents trouvée!")
    
    # Attendre encore un peu pour que tout le contenu soit chargé
    time.sleep(2)
    
    # Récupérer tous les liens avec l'attribut href contenant .pdf
    pdf_links = driver.find_elements(By.XPATH, "//section[@aria-labelledby='documents-title']//a[contains(@href, '.pdf')]")
    
    if pdf_links:
        print(f"\n✓ {len(pdf_links)} lien(s) PDF trouvé(s):\n")
        for i, link in enumerate(pdf_links, 1):
            href = link.get_attribute('href')
            text = link.text.strip()
            print(f"{i}. {text if text else 'Sans titre'}")
            print(f"   → {href}\n")
    else:
        # Si pas de liens PDF, afficher tous les liens de la section
        print("Aucun lien PDF direct trouvé. Affichage de tous les liens de la section:\n")
        all_links = documents_section.find_elements(By.TAG_NAME, "a")
        print(f"{len(all_links)} lien(s) total dans la section:\n")
        for i, link in enumerate(all_links, 1):
            href = link.get_attribute('href')
            text = link.text.strip()
            if href:
                print(f"{i}. {text}")
                print(f"   → {href}\n")
    
except Exception as e:
    print(f"❌ Erreur: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    driver.quit()
    print("\n✓ Driver fermé")


Navigation vers https://basedoc.diplomatie.gouv.fr/pages/details-traites/?refine.doc_ref=TRA20151465...
Attente de 5.1 secondes (comme un humain)...
Page chargée, recherche de la section Documents...
✓ Section Documents trouvée!
Aucun lien PDF direct trouvé. Affichage de tous les liens de la section:

0 lien(s) total dans la section:


✓ Driver fermé


In [6]:
# Fonction pour extraire le lien PDF d'un document
def get_pdf_link(cote, driver, base_url):
    """Récupère le lien PDF pour un document donné"""
    try:
        url = base_url + str(cote)
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        
        # Attendre la section Documents
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section[aria-labelledby='documents-title']")))
        
        # Chercher les liens PDF
        pdf_links = driver.find_elements(By.XPATH, "//section[@aria-labelledby='documents-title']//a[contains(@href, '.pdf')]")
        
        if pdf_links:
            # Retourner le premier lien PDF trouvé
            return pdf_links[0].get_attribute('href')
        return None
    except Exception as e:
        print(f"  ❌ Erreur pour {cote}: {e}")
        return None

# Initialiser le driver en mode headless
options = Options()
options.add_argument('--headless')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Extraire les liens PDF pour tous les documents (limiter à 10 pour tester)
pdf_urls = []
n_docs = min(10, len(tab_doc))  # Limiter à 10 docs pour le test

print(f"Extraction des liens PDF pour {n_docs} documents...\n")

for idx in tab_doc.index[:n_docs]:
    cote = tab_doc.loc[idx, "Cote"]
    print(f"[{idx+1}/{n_docs}] {cote}...", end=" ")
    
    pdf_url = get_pdf_link(cote, driver, base_url)
    pdf_urls.append(pdf_url)
    
    if pdf_url:
        print(f"✓ PDF trouvé")
    else:
        print("✗ Pas de PDF")
    
    # Petit délai entre chaque requête
    time.sleep(random.uniform(1, 3))

driver.quit()

# Ajouter la colonne PDF_URL au DataFrame
tab_doc['PDF_URL'] = None
tab_doc.loc[tab_doc.index[:n_docs], 'PDF_URL'] = pdf_urls

print(f"\n✓ Terminé ! {sum(1 for url in pdf_urls if url)} PDFs trouvés sur {n_docs} documents")
tab_doc[['Cote', 'Titre', 'PDF_URL']].head(n_docs)


Extraction des liens PDF pour 10 documents...

[1/10] TRA20151462... ✓ PDF trouvé
[2/10] TRA20151465... ✓ PDF trouvé
[3/10] TRA20151466... ✓ PDF trouvé
[4/10] TRA20151505... ✓ PDF trouvé
[5/10] TRA20151514... ✗ Pas de PDF
[6/10] TRA20151515... ✗ Pas de PDF
[7/10] TRA20151565... ✓ PDF trouvé
[8/10] TRA20151600... ✓ PDF trouvé
[9/10] TRA20151622... ✗ Pas de PDF
[10/10] TRA20151661... ✗ Pas de PDF

✓ Terminé ! 6 PDFs trouvés sur 10 documents


,Cote,Titre,PDF_URL
0,TRA20151462,Déclaration d'intention entre le ministre des ...,https://fr.ftp.opendatasoft.com/mineae/traites...
1,TRA20151465,Déclaration d'intention entre la ministre de l...,https://fr.ftp.opendatasoft.com/mineae/traites...
2,TRA20151466,Déclaration d'intention entre le ministre des ...,https://fr.ftp.opendatasoft.com/mineae/traites...
3,TRA20151505,Déclaration d'intention dans le domaine du tou...,https://fr.ftp.opendatasoft.com/mineae/traites...
4,TRA20151514,Déclaration d'intention entre la ministre de l...,None
5,TRA20151515,Déclaration d'intention entre la ministre de l...,None
6,TRA20151565,Déclaration d'intention entre la ministre des ...,https://fr.ftp.opendatasoft.com/mineae/traites...
7,TRA20151600,Déclaration d'intention entre la ministre des ...,https://fr.ftp.opendatasoft.com/mineae/traites...
8,TRA20151622,Accord entre l'ambassade de France en Chine et...,None
9,TRA20151661,Déclaration d'intention sur la coopération cin...,None


In [8]:
# Installation: pip install pdfplumber PyMuPDF pytesseract pillow
# PyMuPDF (fitz) est plus facile à installer que pdf2image/poppler

import requests
import io
import pdfplumber
import fitz  
from PIL import Image
import pytesseract

def extract_text_from_pdf(pdf_url, min_text_threshold=100):
    """
    1. Essaie d'abord pdfplumber (extraction texte natif)
    2. Si échec, utilise PyMuPDF pour OCR
    """
    
    print(f"📥 Téléchargement du PDF...")
    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ Erreur téléchargement: {e}")
        return None, "error"
    
    pdf_bytes = io.BytesIO(response.content)
    all_text = ""

    # =========================
    # 1️⃣ Tentative avec pdfplumber
    # =========================
    try:
        print("📄 Tentative extraction avec pdfplumber...")
        with pdfplumber.open(pdf_bytes) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    all_text += text + "\n"

        if len(all_text.strip()) > min_text_threshold:
            print(f"✅ Extraction réussie avec pdfplumber ({len(all_text)} caractères)")
            return all_text, "pdfplumber"
        else:
            print(f"⚠️  Texte trop court ({len(all_text.strip())} caractères) → fallback OCR")

    except Exception as e:
        print(f"❌ Erreur pdfplumber: {e}")
        print("→ Fallback OCR...")

    # =========================
    # 2️⃣ OCR avec PyMuPDF (pas besoin de poppler!)
    # =========================
    all_text = ""
    try:
        print("🔍 OCR avec PyMuPDF...")
        pdf_bytes.seek(0)  # Reset le curseur
        
        # Ouvrir avec PyMuPDF
        pdf_document = fitz.open(stream=pdf_bytes, filetype="pdf")
        
        for page_num in range(len(pdf_document)):
            page = pdf_document[page_num]
            
            # Convertir la page en image (haute résolution)
            pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            
            # OCR avec Tesseract
            text = pytesseract.image_to_string(img, lang="fra+eng")
            all_text += f"\n--- Page {page_num + 1} ---\n{text}"
            
            print(f"  ✓ Page {page_num + 1}/{len(pdf_document)} traitée")
        
        pdf_document.close()
        print(f"✅ Extraction réussie avec OCR ({len(all_text)} caractères)")
        return all_text, "ocr"

    except Exception as e:
        print(f"❌ Erreur OCR: {e}")
        return None, "error"

# Test sur le premier document avec PDF_URL
if 'PDF_URL' in tab_doc.columns and tab_doc.loc[0, 'PDF_URL']:
    pdf_url_test = tab_doc.loc[0, 'PDF_URL']
    print(f"Test sur: {pdf_url_test}\n")
    
    text, method = extract_text_from_pdf(pdf_url_test)
    
    if text:
        print(f"\n📊 Méthode utilisée : {method}")
        print(f"📏 Longueur totale : {len(text)} caractères")
        print(f"\n📝 Aperçu (premiers 1000 caractères) :\n")
        print("="*80)
        print(text[:1000])
        print("="*80)
    else:
        print("\n❌ Échec de l'extraction")
else:
    print("⚠️  Aucun PDF_URL trouvé. Exécutez d'abord la cellule d'extraction des liens.")


ModuleNotFoundError: No module named 'frontend'